# M1 새 생성 — K1 평가 세트 (`m1_fresh/`)

아래는 M1 노트북과 **같은 코드**를 드라이브 `m1_fresh/`에 다시 돌린다. 같은 dev-L 근거·두 모델·같은 흐름으로 **응답을 새로 생성**해 누구도 보지 않은 평가 세트를 만든다(SSOT §6 K1 주 평가). 재보정(M1-2)은 하지 않는다. 출력되는 M1-0·M1-1 판정은 **참고용**이며 M1 결과를 바꾸지 않는다. 기존 조합의 점수는 이 실행에서 저장되고, K1 학습 노트북이 새 판정기 점수를 더한다.

> 런타임 GPU(아무거나), `ANTHROPIC_API_KEY` 필요, 비용 약 $5(M1 실측 $4.95). 끝나면 `m1_fresh_results.zip`을 내려받는다(올릴 필요 없음 — K1 학습 노트북이 드라이브에서 읽음).

# M1 — 최신 LLM·컬렉션 조건 점검 (배포 판정기 임계값)

**왜 하나**: 배포 판정기(I3 조합: F1 base + I2, 임계값 2.992)는 RAGTruth의 **2023년 무렵 LLM 응답**으로 "멀쩡한 응답 10%만 표시"되게 맞췄다. 실제 서비스는 최신 LLM이 답하고, 게이트웨이가 **컬렉션에서 검색한 조각**을 근거로 쓴다. 이 조건에서 멀쩡한 응답에 경고가 너무 많이 뜨면 서비스로 쓸 수 없다.

**무엇을 하나** (학습 없음)
- 근거: **dev-L 근거 문서 전부**(RAGTruth train, 학습·cal-L과 근거가 겹치지 않음)
- 생성: `claude-haiku-5-5`·`claude-sonnet-5-5`(배치, 기본 설정), 근거마다 모델당 1개
  - **QA = 컬렉션 RAG**: dev-L QA 근거를 컬렉션 하나에 등록 → 질문으로 검색(top-4, 120단어 조각) → 게이트웨이 시스템 지시문으로 생성 → 판정 근거 = 검색된 조각
  - **Summary·Data2txt = 근거 직접 전달**: RAGTruth 원 프롬프트, 판정 근거 = 원 근거
- 정답: `claude-opus-5-5`가 R4와 같은 지시문으로 문장마다 S/H/A. **멀쩡한 응답 = H 문장 없음**(C1 규칙)
- 판정: 게이트웨이 `judge.py`로 문장마다 8토큰 + 문장 끝, 조합 점수가 임계값 이상인 지점이 하나라도 있으면 "표시"

**사전 정의 기준** (SSOT §6 M1, 실행 전 고정)
- **M1-0 유효성**: 정답 확보 응답 ≥ 90%, 멀쩡한 응답 ≥ 200개. 미달이면 판정 보류
- **M1-1 (주)**: 두 모델·세 과제를 합친 **멀쩡한 응답 표시율 ≤ 15%**
- **M1-2 (M1-1 미달일 때만 자동 실행)**: cal-L 근거로 같은 흐름 → 그 멀쩡한 응답으로 서비스 10% 임계값 재보정 → dev-L 멀쩡한 응답 표시율 ≤ 15%면 "재보정 임계값 후보"(교체는 별도 결정)
- 보고만: 모델·과제별 표시율(95% 구간), F1 base 단독, 엄격 멀쩡(모든 문장 S), 환각 응답 탐지율, 표시된 응답 중 실제 환각 비율, QA 검색에 원래 근거가 들어간 비율

**실행 방법**
1. 왼쪽 열쇠 아이콘(보안 비밀)에 `ANTHROPIC_API_KEY`를 넣고 노트북 접근을 켠다
2. 런타임 유형: **GPU**(A100·L4·T4 아무거나. 지연은 재지 않음)
3. **모두 실행**. 배치는 보통 수십 분~1시간. 끊기면 다시 모두 실행 — 제출한 배치·받은 결과·판정 점수는 드라이브 `m1/`에서 이어서 한다
4. 비용(추정, 실행 중 출력): dev-L 약 $5~8. M1-1 미달이면 cal-L도 돌아 약 2배
5. 끝나면 `m1_results.zip`이 내려받아진다


## 1. 환경 준비

In [ ]:
SRC = {'judge': '"""문장 판정기.\n\n입력 방식은 판정기 폴더의 `judge_config.json`이 정한다 (SPEC §4).\n  retrieval (F1 base): 근거 → 40단어 조각 → 임베딩 (요청당 1회, prepare)\n                       판정 텍스트 → 임베딩 → 유사한 조각 top-k (원문 순서) → 교차 인코더 [근거 조각, 문장 앞부분]\n  full      (I2식)    : 교차 인코더 [근거 전체, 응답 처음부터 판정 지점까지]. 근거 검색 없음\n점수는 logit(클수록 환각). 임계값은 재판정 간격(stride)별로 설정 파일의 값을 쓴다.\n"""\nimport json\nimport os\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nimport torch\n\n\n@dataclass\nclass Verdict:\n    score: float          # logit, 클수록 환각\n    hallucinated: bool\n    judge_ms: float       # 검색 + 판정 시간\n\n\n@dataclass\nclass RefIndex:\n    chunks: list\n    emb: object           # torch.Tensor [n_chunks, d] (retrieval) / None (full)\n    prepare_ms: float\n    text: str = \'\'        # 근거 원문 (full 방식)\n\n\ndef chunk_text(text, n):\n    w = text.split()\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]\n\n\ndef _stride_key(stride):\n    return \'end\' if not stride else str(stride)\n\n\ndef read_config(model_dir):\n    p = os.path.join(model_dir, \'judge_config.json\')\n    return json.load(open(p)) if os.path.isfile(p) else {}\n\n\ndef input_mode(cfg):\n    """설정 파일에서 입력 방식을 정한다. 명시(`input_mode`)가 없으면 I2 노트북 형식(retrieval: null)을 full로 본다."""\n    if cfg.get(\'input_mode\'):\n        return cfg[\'input_mode\']\n    inp = cfg.get(\'input\') or {}\n    return \'full\' if isinstance(inp, dict) and \'retrieval\' in inp and inp[\'retrieval\'] is None else \'retrieval\'\n\n\nclass StudentJudge:\n    def __init__(self, model_dir, embedder=\'sentence-transformers/all-MiniLM-L6-v2\', k=8, max_len=512,\n                 chunk_words=40, threshold=0.0, device=\'\', mode=\'retrieval\', thresholds=None, threshold_key=\'svc10_orig\',\n                 default_stride=0):\n        from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.mode = mode\n        self.tok = AutoTokenizer.from_pretrained(model_dir)\n        cfg = AutoConfig.from_pretrained(model_dir)\n        kw = {}\n        if getattr(cfg, \'model_type\', \'\') == \'modernbert\':\n            # 새 transformers는 reference_compile을 모델 인자로 받지 않으므로 설정에 넣는다\n            if hasattr(cfg, \'reference_compile\'):\n                cfg.reference_compile = False\n            kw[\'attn_implementation\'] = \'sdpa\'\n        # 가중치가 fp16으로 저장돼 있어도 fp32로 올린다 (CPU 판정, 노트북 점수와 일치). GPU에서는 autocast가 반정밀도로 돌린다.\n        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir, config=cfg, **kw).float().to(self.device).eval()\n        if mode == \'retrieval\' and isinstance(embedder, str):\n            from sentence_transformers import SentenceTransformer\n            embedder = SentenceTransformer(embedder, device=self.device)\n        self.embedder = embedder\n        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold\n        self.thresholds, self.threshold_key = thresholds or {}, threshold_key\n        self.default_stride = default_stride\n        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화\n\n    def threshold_for(self, stride, key=None):\n        """재판정 간격별 임계값 (설정 파일 thresholds[간격][key]). 없으면 기본 임계값."""\n        return (self.thresholds.get(_stride_key(stride)) or {}).get(key or self.threshold_key, self.threshold)\n\n    def _embed(self, texts):\n        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,\n                                    show_progress_bar=False)\n\n    def prepare(self, reference):\n        t0 = time.perf_counter()\n        if self.mode == \'full\':\n            return RefIndex([], None, (time.perf_counter() - t0) * 1000, reference)\n        chunks = chunk_text(reference, self.chunk_words)\n        with self._lock:\n            emb = self._embed(chunks)\n        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000, reference)\n\n    def _pair(self, ref, text, prior):\n        if self.mode == \'full\':\n            return ref.text, (prior or \'\') + text\n        e = self._embed([text])[0]\n        top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()\n        return \' ... \'.join(ref.chunks[i] for i in sorted(top)), text\n\n    @torch.no_grad()\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        """text: 판정할 문장(또는 문장 앞부분, 앞뒤 공백 제거), prior: 응답에서 그 문장 앞까지의 원문."""\n        t0 = time.perf_counter()\n        with self._lock:\n            a, b = self._pair(ref, text, prior)\n            enc = self.tok([a], [b], truncation=\'only_first\', max_length=self.max_len,\n                           return_tensors=\'pt\').to(self.device)\n            with torch.autocast(self.device, enabled=(self.device == \'cuda\')):\n                score = float(self.model(**enc).logits.squeeze().float().cpu())\n            if self.device == \'cuda\':\n                torch.cuda.synchronize()\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n\n\ndef load_judge(settings):\n    """판정기 폴더의 judge_config.json을 읽어 StudentJudge를 만든다. 환경 변수로 지정한 값이 설정 파일보다 우선한다."""\n    cfg = read_config(settings.judge_model_dir)\n    pick = lambda env, key, default: env if env is not None else cfg.get(key, default)\n    dflt = (cfg.get(\'default\') or {})\n    kind = dflt.get(\'kind\', \'end\')\n    return StudentJudge(settings.judge_model_dir,\n                        embedder=pick(settings.judge_embedder, \'embedder\', \'sentence-transformers/all-MiniLM-L6-v2\'),\n                        k=pick(settings.judge_k, \'k\', 8), max_len=pick(settings.judge_max_len, \'max_len\', 512),\n                        chunk_words=pick(settings.judge_chunk_words, \'chunk_words\', 40),\n                        threshold=pick(settings.judge_threshold, \'threshold\', 0.0), device=settings.judge_device,\n                        mode=input_mode(cfg),\n                        # 임계값을 환경 변수로 고정하면 간격별 임계값은 쓰지 않는다\n                        thresholds={} if settings.judge_threshold is not None else cfg.get(\'thresholds\', {}),\n                        threshold_key=dflt.get(\'threshold\', \'svc10_orig\'),\n                        default_stride=0 if kind == \'end\' else int(kind))\n\n\nclass LexicalJudge:\n    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).\n    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""\n\n    default_stride = 0\n\n    def __init__(self, threshold=0.5, delay_ms=0.0, fail=None):\n        self.threshold, self.delay_ms = threshold, delay_ms\n        self.fail = fail          # 테스트용: 이 함수가 True를 돌려주는 판정에서 예외를 낸다\n\n    def threshold_for(self, stride, key=None):\n        return self.threshold\n\n    def prepare(self, reference):\n        return RefIndex([reference], {w.lower().strip(\'.,\') for w in reference.split()}, 0.0, reference)\n\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        t0 = time.perf_counter()\n        if self.fail is not None and self.fail(text):\n            raise RuntimeError(\'judge failure (test)\')\n        if self.delay_ms:\n            time.sleep(self.delay_ms / 1000)\n        words = [w.lower().strip(\'.,\') for w in text.split() if w.strip(\'.,\')]\n        score = sum(w not in ref.emb for w in words) / max(len(words), 1)\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n', 'store': '"""컬렉션 저장소 (SPEC §2.5, §2.6).\n\n관리자·개발자가 등록한 문서를 조각으로 나눠 임베딩과 함께 저장하고, 질문으로 검색한다.\n게이트웨이는 찾은 조각을 LLM 프롬프트에 넣고, **같은 조각을 판정 근거로** 쓴다(LLM이 본 근거 = 판정 근거).\n\n  Store 인터페이스\n    add(collection, text, title=\'\', doc_id=None) -> dict(doc_id, title, n_chunks, n_chars)\n    list(collection) -> [dict(doc_id, title, n_chunks, n_chars, created)]\n    delete(collection, doc_id) -> bool\n    collections() -> [dict(collection, n_docs)]\n    search(collection, query, k) -> [Evidence]       # 점수 내림차순\n\n처음 구현은 SQLite 파일 하나 + 임베딩 배열(코사인 유사도 전수 검색)이다. 규모가 커지면 같은 인터페이스로 벡터 DB로 바꾼다.\n임베더는 encode(texts) -> 정규화된 벡터 배열을 돌려주는 객체(sentence-transformers 호환)면 된다.\n"""\nimport sqlite3\nimport threading\nimport time\nimport uuid\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\n\nSCHEMA = \'\'\'\nCREATE TABLE IF NOT EXISTS documents (\n    collection TEXT NOT NULL, doc_id TEXT NOT NULL, title TEXT, text TEXT NOT NULL, created REAL NOT NULL,\n    PRIMARY KEY (collection, doc_id));\nCREATE TABLE IF NOT EXISTS chunks (\n    collection TEXT NOT NULL, doc_id TEXT NOT NULL, idx INTEGER NOT NULL, text TEXT NOT NULL, emb BLOB NOT NULL,\n    PRIMARY KEY (collection, doc_id, idx));\n\'\'\'\n\n\n@dataclass\nclass Evidence:\n    doc_id: str\n    title: str\n    chunk_idx: int\n    text: str\n    score: float\n\n    def to_dict(self):\n        return asdict(self)\n\n\ndef split_words(text, n, overlap=0):\n    """n단어 조각(겹침 overlap단어). 빈 문서는 조각 없음."""\n    w = text.split()\n    if not w:\n        return []\n    step = max(1, n - overlap)\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w) - overlap, 1), step)]\n\n\nclass SQLiteStore:\n    def __init__(self, path, embedder, chunk_words=120, overlap=20):\n        self.path, self.embedder = path, embedder\n        self.chunk_words, self.overlap = chunk_words, overlap\n        self._db = sqlite3.connect(path, check_same_thread=False)\n        self._db.executescript(SCHEMA)\n        self._lock = threading.Lock()\n        self._cache = {}          # collection -> (meta 리스트, 임베딩 행렬)\n\n    def _embed(self, texts):\n        v = self.embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)\n        return np.asarray(v, dtype=np.float32).reshape(len(texts), -1)\n\n    def add(self, collection, text, title=\'\', doc_id=None):\n        if not text or not text.strip():\n            raise ValueError(\'document text is empty\')\n        doc_id = doc_id or uuid.uuid4().hex[:16]\n        chunks = split_words(text, self.chunk_words, self.overlap)\n        emb = self._embed(chunks)\n        with self._lock, self._db:\n            self._db.execute(\'DELETE FROM chunks WHERE collection=? AND doc_id=?\', (collection, doc_id))   # 같은 id면 교체\n            self._db.execute(\'INSERT OR REPLACE INTO documents VALUES (?,?,?,?,?)\',\n                             (collection, doc_id, title or \'\', text, time.time()))\n            self._db.executemany(\'INSERT INTO chunks VALUES (?,?,?,?,?)\',\n                                 [(collection, doc_id, i, c, e.tobytes()) for i, (c, e) in enumerate(zip(chunks, emb))])\n            self._cache.pop(collection, None)\n        return {\'doc_id\': doc_id, \'title\': title or \'\', \'n_chunks\': len(chunks), \'n_chars\': len(text)}\n\n    def list(self, collection):\n        with self._lock:\n            rows = self._db.execute(\n                \'SELECT d.doc_id, d.title, length(d.text), d.created, \'\n                \'(SELECT count(*) FROM chunks c WHERE c.collection=d.collection AND c.doc_id=d.doc_id) \'\n                \'FROM documents d WHERE d.collection=? ORDER BY d.created\', (collection,)).fetchall()\n        return [{\'doc_id\': r[0], \'title\': r[1], \'n_chars\': r[2], \'created\': r[3], \'n_chunks\': r[4]} for r in rows]\n\n    def delete(self, collection, doc_id):\n        with self._lock, self._db:\n            n = self._db.execute(\'DELETE FROM documents WHERE collection=? AND doc_id=?\', (collection, doc_id)).rowcount\n            self._db.execute(\'DELETE FROM chunks WHERE collection=? AND doc_id=?\', (collection, doc_id))\n            self._cache.pop(collection, None)\n        return n > 0\n\n    def collections(self):\n        with self._lock:\n            rows = self._db.execute(\'SELECT collection, count(*) FROM documents GROUP BY collection ORDER BY collection\').fetchall()\n        return [{\'collection\': c, \'n_docs\': n} for c, n in rows]\n\n    def exists(self, collection):\n        with self._lock:\n            return self._db.execute(\'SELECT 1 FROM documents WHERE collection=? LIMIT 1\', (collection,)).fetchone() is not None\n\n    def _matrix(self, collection):\n        with self._lock:\n            if collection not in self._cache:\n                rows = self._db.execute(\n                    \'SELECT c.doc_id, d.title, c.idx, c.text, c.emb FROM chunks c JOIN documents d \'\n                    \'ON c.collection=d.collection AND c.doc_id=d.doc_id WHERE c.collection=? ORDER BY d.created, c.idx\',\n                    (collection,)).fetchall()\n                meta = [r[:4] for r in rows]\n                mat = np.stack([np.frombuffer(r[4], dtype=np.float32) for r in rows]) if rows else np.zeros((0, 1), np.float32)\n                self._cache[collection] = (meta, mat)\n            return self._cache[collection]\n\n    def search(self, collection, query, k):\n        meta, mat = self._matrix(collection)\n        if not meta or not query.strip():\n            return []\n        sims = mat @ self._embed([query])[0]\n        top = np.argsort(-sims)[:k]\n        return [Evidence(meta[i][0], meta[i][1], meta[i][2], meta[i][3], float(sims[i])) for i in top]\n\n    def close(self):\n        self._db.close()\n', 'context': '"""요청에서 근거 문서(RAG 컨텍스트)와 검증 옵션을 꺼낸다.\n\n우선순위\n1. 최상위 필드 `verification.context` (문자열 또는 문자열 리스트)\n   - OpenAI SDK에서는 `extra_body={"verification": {...}}` 로 보낸다.\n2. 메시지 본문 안의 `<context>...</context>` 태그 (RAG 서비스가 흔히 쓰는 형식)\n3. `verification.collection` — 게이트웨이에 등록된 컬렉션에서 검색 (app이 처리, SPEC §2.6)\n4. 모두 없으면 검증하지 않는다 (mode=off).\n\n`verification` 필드는 실제 LLM API가 모르는 필드이므로 전달 전에 제거한다.\n"""\nimport copy\nimport re\n\nCTX_TAG = re.compile(r\'<context>(.*?)</context>\', re.S | re.I)\nMODES = {\'pass\', \'hold\', \'off\'}\nACTIONS = {\'flag\', \'cut\'}\n\n\ndef _content_text(content):\n    if isinstance(content, str):\n        return content\n    if isinstance(content, list):          # [{"type":"text","text":...}, ...]\n        return \'\\n\'.join(p.get(\'text\', \'\') for p in content if isinstance(p, dict))\n    return \'\'\n\n\nRAG_INSTRUCTION = (\'Answer using only the reference documents below. \'\n                   \'If they do not contain the answer, say that you do not know.\')\n\n\ndef last_user_text(body):\n    users = [m for m in body.get(\'messages\', []) if m.get(\'role\') == \'user\']\n    return _content_text(users[-1].get(\'content\')) if users else \'\'\n\n\ndef rag_messages(messages, evidence):\n    """근거 조각을 담은 시스템 메시지를 맨 앞에 더한다. 서비스가 보낸 메시지는 바꾸지 않는다 (SPEC §2.6)."""\n    block = \'\\n\\n\'.join(f\'[{i + 1}] {e.text}\' for i, e in enumerate(evidence))\n    return [{\'role\': \'system\', \'content\': f\'{RAG_INSTRUCTION}\\n\\n<documents>\\n{block}\\n</documents>\'}] + list(messages)\n\n\ndef rag_reference(evidence):\n    """판정 근거 = LLM에 넣은 조각과 같은 텍스트."""\n    return \'\\n\\n\'.join(e.text for e in evidence)\n\n\ndef extract(body, default_mode=\'pass\', default_action=\'flag\', default_stride=0, with_stride=False, with_opts=False):\n    """반환: (upstream으로 보낼 body, reference 문자열 또는 None, mode, action[, stride][, opts])\n    with_stride=True면 앞부분 재판정 간격(stride)도 함께 돌려준다.\n    with_opts=True면 원래 verification 옵션도 돌려준다(collection 등). collection이 있고 근거가 없으면 mode를 off로 바꾸지 않는다."""\n    body = copy.deepcopy(body)\n    opts = body.pop(\'verification\', None) or {}\n\n    ref = opts.get(\'context\')\n    if isinstance(ref, list):\n        ref = \'\\n\\n\'.join(str(x) for x in ref)\n    if not ref:\n        found = []\n        for m in body.get(\'messages\', []):\n            found += CTX_TAG.findall(_content_text(m.get(\'content\')))\n        ref = \'\\n\\n\'.join(s.strip() for s in found) or None\n\n    mode = opts.get(\'mode\', default_mode)\n    action = opts.get(\'action\', default_action)\n    if mode not in MODES:\n        raise ValueError(f\'verification.mode must be one of {sorted(MODES)}\')\n    if action not in ACTIONS:\n        raise ValueError(f\'verification.action must be one of {sorted(ACTIONS)}\')\n    stride = opts.get(\'stride\', default_stride)\n    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 0:\n        raise ValueError(\'verification.stride must be a non-negative integer\')\n    coll = opts.get(\'collection\')\n    if coll is not None and (not isinstance(coll, str) or not coll):\n        raise ValueError(\'verification.collection must be a non-empty string\')\n    if not ref and not coll:\n        mode = \'off\'\n    out = (body, ref, mode, action) + ((stride,) if with_stride else ()) + ((opts,) if with_opts else ())\n    return out\n'}
import os, sys, json, time, random, subprocess, shutil, pickle
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'anthropic', 'sentence-transformers', 'tiktoken', 'pysbd', '-U', 'transformers>=4.48'], check=True)
subprocess.run('git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || true', shell=True)
import numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')
# 게이트웨이 코드 그대로 (패키지 __init__은 FastAPI 앱을 불러오므로 비워 둔다)
os.makedirs('/content/hg/hallugate', exist_ok=True)
open('/content/hg/hallugate/__init__.py', 'w').write('')
for n, s in SRC.items(): open(f'/content/hg/hallugate/{n}.py', 'w').write(s)
sys.path.insert(0, '/content/hg')
from hallugate.judge import StudentJudge, read_config, input_mode
from hallugate.store import SQLiteStore
from hallugate.context import rag_messages, rag_reference

OUT = '/content/drive/MyDrive/Colab Notebooks/HalluGate'
F_DIR, I_DIR = os.path.join(OUT, 'final', 'judge_base'), os.path.join(OUT, 'i2', 'judge_i2')
R0_DIR, M1_DIR = os.path.join(OUT, 'r0'), os.path.join(OUT, 'm1_fresh')
os.makedirs(M1_DIR, exist_ok=True)
for d in [F_DIR, I_DIR]:
    assert os.path.exists(os.path.join(d, 'judge_config.json')), f'{d}/judge_config.json 없음'
assert os.path.exists(os.path.join(R0_DIR, 'r0_items.pkl')), 'r0/r0_items.pkl 없음 (cal-L·dev-L 분할)'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
COMBO = {'F_mean': -3.6794357299804688, 'F_std': 2.1870224475860596, 'I_mean': -4.456174373626709, 'I_std': 2.400838851928711, 'threshold': 2.991638706756592}
def save_json(o, p): json.dump(o, open(p + '.tmp', 'w'), ensure_ascii=False); os.replace(p + '.tmp', p)
def load_json(p): return json.load(open(p)) if os.path.exists(p) else None
print('device', GPU, '| 조합 상수(I3)', COMBO)

## 2. 근거 문서 — dev-L(평가), cal-L(M1-2 재보정용)

In [ ]:
# 판정 입력: RAGTruth train 응답의 문장 끝 + 8토큰 지점 (게이트웨이와 같은 cl100k 토큰, 문장 시작부터)
import pysbd, tiktoken
ENC = tiktoken.get_encoding('cl100k_base')
SEG = pysbd.Segmenter(language='en', clean=False, char_span=True)

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

def points_of(text, s=8):
    # (문장 시작 글자, 지점 글자 끝) 목록: 문장마다 s토큰째 + 문장 끝
    ids = ENC.encode(text, disallowed_special=())
    _, starts = ENC.decode_with_offsets(ids)
    ends = starts[1:] + [len(text)]
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        tok = [e for st, e in zip(starts, ends) if a <= st < sp.end or (st < a < e)]
        for k, e in enumerate(tok, 1):
            if k % s == 0 or k == len(tok):
                out.append((a, min(e, sp.end)))
    return out

def make_inputs(responses, sources, n_resp=300, n_pts=300, seed=2027):
    rs = random.Random(seed)
    pool = [r for r in responses if r['split'] == 'train']
    cands = []
    for r in rs.sample(pool, min(n_resp, len(pool))):
        ref = build_reference(sources[r['source_id']])
        for a, cut in points_of(r['response']):
            text = r['response'][a:cut].strip()
            if text: cands.append({'ref': ref, 'text': text, 'prior': r['response'][:a], 'task': sources[r['source_id']]['task_type']})
    return rs.sample(cands, min(n_pts, len(cands)))

BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
SIDS = {sp: sorted({x['ref_key'] for x in R0P['items'][k]}) for sp, k in [('devL', 'devL'), ('calL', 'calL')]}
assert not set(SIDS['devL']) & set(SIDS['calL']), 'dev-L·cal-L 근거가 겹침'
split_of = {}
for r in responses: split_of.setdefault(r['source_id'], set()).add(r['split'])
for sp, ss in SIDS.items():
    assert all(split_of[s] == {'train'} for s in ss), f'{sp}에 test 분할 근거가 섞임'
    print(sp, f'근거 {len(ss)}개 |', {t: sum(sources[s]['task_type'] == t for s in ss) for t in ['QA', 'Summary', 'Data2txt']})

## 3. 판정기·생성 요청 준비

In [ ]:
import contextlib, threading
from concurrent.futures import ThreadPoolExecutor

def make_judge(d, device):
    # load_judge와 같은 값으로 (판정기 폴더의 judge_config.json)
    cfg = read_config(d)
    return StudentJudge(d, embedder=cfg.get('embedder', 'sentence-transformers/all-MiniLM-L6-v2'), k=cfg.get('k', 8),
                        max_len=cfg.get('max_len', 512), chunk_words=cfg.get('chunk_words', 40), device=device,
                        mode=input_mode(cfg))

def combine(sf, si):
    # I3 조합: 두 점수를 cal-L 멀쩡한 응답 기준으로 표준화해 평균
    return ((sf - COMBO['F_mean']) / COMBO['F_std'] + (si - COMBO['I_mean']) / COMBO['I_std']) / 2

def judge_seq(jF, jI, refF, refI, x):
    # 차례로: 게이트웨이 judge() 두 번 (각각 토크나이즈·검색·순전파·동기화 포함)
    return jF.judge(refF, x['text'], x['prior']).score, jI.judge(refI, x['text'], x['prior']).score

STREAMS = {}
def _score_on_stream(j, ref, text, prior):
    # judge()와 같은 계산을 판정기 전용 CUDA 스트림에서. 동기화는 그 스트림만 (.cpu()가 현재 스트림을 기다림)
    ctx = torch.cuda.stream(STREAMS.setdefault(id(j), torch.cuda.Stream())) if j.device == 'cuda' else contextlib.nullcontext()
    with ctx, torch.no_grad():
        a, b = j._pair(ref, text, prior)
        enc = j.tok([a], [b], truncation='only_first', max_length=j.max_len, return_tensors='pt').to(j.device)
        with torch.autocast(j.device, enabled=(j.device == 'cuda')):
            return float(j.model(**enc).logits.squeeze().float().cpu())

POOL = ThreadPoolExecutor(2)
def judge_par(jF, jI, refF, refI, x):
    fF = POOL.submit(_score_on_stream, jF, refF, x['text'], x['prior'])
    fI = POOL.submit(_score_on_stream, jI, refI, x['text'], x['prior'])
    return fF.result(), fI.result()

def bench(fn, jF, jI, inputs, refs, warm=10):
    for x in inputs[:warm]: fn(jF, jI, *refs[x['ref']], x)
    ms, sc = [], []
    for x in inputs:
        t0 = time.perf_counter()
        sf, si = fn(jF, jI, *refs[x['ref']], x)
        c = combine(sf, si)
        ms.append((time.perf_counter() - t0) * 1000); sc.append((sf, si, c))
    return np.array(ms), np.array(sc)

def bench_one(j, inputs, refs, which, warm=10):
    for x in inputs[:warm]: j.judge(refs[x['ref']][which], x['text'], x['prior'])
    return np.array([j.judge(refs[x['ref']][which], x['text'], x['prior']).judge_ms for x in inputs])

# 생성 요청 만들기: QA = 컬렉션 RAG(게이트웨이 store·context 그대로), Summary·Data2txt = 근거 직접 전달(RAGTruth 원 프롬프트)
GENS = {'haiku': 'claude-haiku-5-5', 'sonnet': 'claude-sonnet-5-5'}
LABELER = 'claude-opus-5-5'
TOP_K, CHUNK_W, OVERLAP = 4, 120, 20          # 게이트웨이 기본값 (config.py)
GEN_MAX_TOKENS = 8000

def build_collection(store, coll, sids, sources):
    n = 0
    for sid in sids:
        if sources[sid]['task_type'] != 'QA': continue
        text = sources[sid]['source_info'].get('passages', '')
        if text.strip():
            store.add(coll, text, title=sid, doc_id=sid); n += 1
    return n

def make_cases(split, sids, sources, store, coll):
    cases = {}
    for sid in sids:
        src = sources[sid]; tt = src['task_type']
        if tt == 'QA':
            q = src['source_info'].get('question', '')
            ev = store.search(coll, q, TOP_K)
            msgs = rag_messages([{'role': 'user', 'content': q}], ev)
            system, messages = msgs[0]['content'], msgs[1:]          # OpenAI 형식 시스템 메시지 → Messages API system
            ref_j = rag_reference(ev)
            extra = {'flow': 'collection', 'gold_in': any(e.doc_id == sid for e in ev),
                     'ref_label': f'Question: {q}\n{ref_j}'}
        else:
            system, messages = None, [{'role': 'user', 'content': src['prompt']}]
            ref_j = build_reference(src)
            extra = {'flow': 'direct', 'gold_in': None, 'ref_label': ref_j}
        for g, model in GENS.items():
            cases[f'{split}-{g}-{sid}'] = {'sid': sid, 'task': tt, 'gen': g, 'model': model, 'system': system,
                                           'messages': messages, 'ref_judge': ref_j, **extra}
    return cases

def gen_params(c):
    p = {'model': c['model'], 'max_tokens': GEN_MAX_TOKENS, 'messages': c['messages']}
    if c['system']: p['system'] = c['system']
    return p
SYSTEM = '''You are an expert annotator for RAGTruth, a benchmark of hallucinations in retrieval-augmented generation.
You receive a REFERENCE that a language model was given, and the model's RESPONSE split into numbered sentences.
Label EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution).
- Evident baseless info: introduces information that is not in the reference, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment (e.g. "popular", "cozy", "family-friendly") that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization; statements that the reference lacks the information or declining to answer;
meta phrases such as "Based on the passages,". For Data2txt (business data + user reviews), claims supported by the structured data or by what reviewers say are supported.

Labels: "S" = no hallucination. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give: "type" (one of: evident_conflict, subtle_conflict, evident_baseless, subtle_baseless),
"phrase" (the shortest span copied EXACTLY, character for character, from that sentence that is hallucinated), and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''

TASK_DESC = {'QA': 'QA — answer the question using the passages',
             'Summary': 'Summary — summarize the news article',
             'Data2txt': 'Data2txt — write an overview of a business from structured data and user reviews'}

def sentences(text):
    # (시작, 끝, 문장) — 판정 지점(points_of)과 같은 pysbd 분할
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        out.append((a, sp.start + len(sp.sent.rstrip()), sp.sent.strip()))
    return out

def label_msg(c, text):
    ss = sentences(text)
    lines = '\n'.join(f'[{i + 1}] {s}' for i, (_, _, s) in enumerate(ss))
    return f"TASK: {TASK_DESC[c['task']]}\n\nREFERENCE:\n<<<\n{c['ref_label']}\n>>>\n\nRESPONSE ({len(ss)} numbered sentences):\n{lines}"

def label_params(c, text):
    # R4와 같은 요청 (모델·max_tokens·지시문)
    return {'model': LABELER, 'max_tokens': 4096, 'system': SYSTEM, 'messages': [{'role': 'user', 'content': label_msg(c, text)}]}

def parse(text, n):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        labs = {}
        for e in d.get('sentences', []):
            i, lab = int(e.get('i', 0)), str(e.get('label', '')).strip().upper()[:1]
            if 1 <= i <= n and lab and lab in 'SHA':
                labs[i] = {'label': lab, 'type': e.get('type', ''), 'phrase': e.get('phrase', '') or '', 'reason': e.get('reason', '')}
        return labs, len(labs) == n
    except Exception:
        return {}, False

def score_response(jF, jI, refs, text, fn):
    # 판정 지점(문장마다 8토큰째 + 문장 끝)마다 (문장 시작, 지점 끝, F1 base, I2, 조합)
    out = []
    for a, cut in points_of(text):
        t = text[a:cut].strip()
        if not t: continue
        sf, si = fn(jF, jI, refs[0], refs[1], {'text': t, 'prior': text[:a]})
        out.append((a, cut, sf, si, combine(sf, si)))
    return out

def gen_status(g):
    if g is None: return 'nogen'
    if g['stop'] == 'refusal': return 'refusal'
    if not g['text'].strip() or not sentences(g['text']): return 'empty'
    return 'valid'

jF, jI = make_judge(F_DIR, DEVICE), make_judge(I_DIR, DEVICE)
assert jF.mode == 'retrieval' and jI.mode == 'full', (jF.mode, jI.mode)
THR_C = COMBO['threshold']
THR_F = read_config(F_DIR)['thresholds']['8']['svc10_orig']
FN = judge_par if DEVICE == 'cuda' else judge_seq
print(f'임계값: 조합 {THR_C:.4f} | F1 base 단독 {THR_F:.4f} (보고용)')

def prepare_split(split):
    cp = os.path.join(M1_DIR, f'cases_{split}.json')
    cases = load_json(cp)                      # 이미 만든 요청이 있으면 그대로 (검색 결과 = LLM이 본 근거가 바뀌지 않게)
    if cases is None:
        store = SQLiteStore(f'/content/m1_{split}.sqlite3', jF.embedder, CHUNK_W, OVERLAP)
        coll = f'm1-{split}'
        if not store.exists(coll):
            print(f'컬렉션 {coll}: QA 근거 {build_collection(store, coll, SIDS[split], sources)}개 등록')
        cases = make_cases(split, SIDS[split], sources, store, coll)
        save_json(cases, cp)
    qa = [c for c in cases.values() if c['flow'] == 'collection' and c['gen'] == 'haiku']
    print(f"{split}: 생성 요청 {len(cases)}개 | QA 검색에 원래 근거가 들어간 비율 {np.mean([c['gold_in'] for c in qa]):.3f}")
    return cases

CASES = {'devL': prepare_split('devL')}

## 4. Claude API — 형식 확인 3건, 배치 제출·결과 받기 (드라이브에서 이어서)

In [ ]:
import anthropic
def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception:
        pass
    from getpass import getpass
    return getpass(f'{name}: ')
A = anthropic.Anthropic(api_key=get_key('ANTHROPIC_API_KEY'), max_retries=3)
PRICE = {'claude-haiku-5-5': (0.05, 0.25), 'claude-sonnet-5-5': (1.0, 5.0), 'claude-opus-5-5': (2.0, 10.0)}   # 배치 $/백만 토큰 (2026-11)

def text_of(m): return ''.join(b.text for b in m.content if getattr(b, 'type', '') == 'text')
def rec(m): return {'text': text_of(m), 'stop': m.stop_reason, 'model': m.model, 'in': m.usage.input_tokens, 'out': m.usage.output_tokens}
def cost(res): return sum(r['in'] / 1e6 * PRICE.get(r['model'], (2, 10))[0] + r['out'] / 1e6 * PRICE.get(r['model'], (2, 10))[1] for r in res.values())

def run_batch(reqs, path, chunk=400):
    # reqs: cid -> 요청. 결과: cid -> rec. 다시 실행하면 제출한 묶음·받은 결과는 건너뛴다
    res = load_json(path) or {}
    bpath = path.replace('.json', '_batches.json')
    B = load_json(bpath) or {'chunks': []}
    sent = {c for ch in B['chunks'] for c in ch['cids']}
    todo = [c for c in reqs if c not in sent and c not in res]
    for k in range(0, len(todo), chunk):
        part = todo[k:k + chunk]
        b = A.messages.batches.create(requests=[{'custom_id': c, 'params': reqs[c]} for c in part])
        B['chunks'].append({'id': b.id, 'cids': part, 'done': False}); save_json(B, bpath)
        print(f'  제출 {b.id} ({len(part)}건)', flush=True)
    for ch in B['chunks']:
        if ch.get('done'): continue
        while True:
            b = A.messages.batches.retrieve(ch['id']); rc = b.request_counts
            print(f'  {time.strftime("%H:%M:%S")} {ch["id"][-8:]} {b.processing_status} | 처리 중 {rc.processing}, 성공 {rc.succeeded}, 오류 {rc.errored}', flush=True)
            if b.processing_status == 'ended': break
            time.sleep(60)
        for r in A.messages.batches.results(ch['id']):
            if r.result.type == 'succeeded': res[r.custom_id] = rec(r.result.message)
        ch['done'] = True; save_json(B, bpath); save_json(res, path)
    miss = [c for c in reqs if c not in res]
    for c in miss[:100]:                                   # 배치 실패분은 바로 요청 1회
        try: res[c] = rec(A.messages.create(**reqs[c]))
        except Exception as e: print(f'  {c}: {type(e).__name__} {str(e)[:150]}'); break
    save_json(res, path)
    print(f'  결과 {len(res)}/{len(reqs)} | 사용량 약 ${cost({c: res[c] for c in reqs if c in res}):.2f}')
    return res

# 형식 확인: 모델마다 1건 바로 생성 + 그 응답 정답 1건 (수 센트)
for g in GENS:
    cid = next(c for c, x in CASES['devL'].items() if x['gen'] == g and x['flow'] == 'collection')
    m = A.messages.create(**gen_params(CASES['devL'][cid])); r = rec(m)
    print(f"[{g}] stop={r['stop']} 입력 {r['in']} 출력 {r['out']} | {r['text'][:200]!r}")
    if r['text'].strip():
        lm = A.messages.create(**label_params(CASES['devL'][cid], r['text'])); labs, ok = parse(text_of(lm), len(sentences(r['text'])))
        print(f"   정답 파싱 {'성공' if ok else '실패'} | {[v['label'] for v in labs.values()]}")
n = len(CASES['devL'])
print(f'dev-L 예상 비용: 생성 약 ${n / 2 * (1500 * 0.05 + 1500 * 0.25 + 1500 * 1.0 + 1500 * 5.0) / 1e6:.1f} + 정답 약 ${n * (2200 * 2.0 + 700 * 10.0) / 1e6:.1f} (응답당 토큰 추정)')

## 5. 생성 → 정답 → 판정 (dev-L)

In [ ]:
def run_split(split):
    cases = CASES[split]
    gens = run_batch({c: gen_params(x) for c, x in cases.items()}, os.path.join(M1_DIR, f'gen_{split}.json'))
    valid = [c for c in cases if gen_status(gens.get(c)) == 'valid']
    print(f'{split} 생성: 유효 {len(valid)} | 거절 {sum(gen_status(gens.get(c)) == "refusal" for c in cases)} | '
          f'빈 응답 {sum(gen_status(gens.get(c)) == "empty" for c in cases)} | max_tokens {sum((gens.get(c) or {}).get("stop") == "max_tokens" for c in cases)}')
    raw = run_batch({c: label_params(cases[c], gens[c]['text']) for c in valid}, os.path.join(M1_DIR, f'label_{split}.json'))
    labels, bad = {}, []
    for c in valid:
        if c not in raw: continue
        labs, ok = parse(raw[c]['text'], len(sentences(gens[c]['text'])))
        (labels.__setitem__(c, labs) if ok else bad.append(c))
    for c in bad[:50]:                                     # 파싱 실패는 바로 1회 재요청
        try:
            labs, ok = parse(text_of(A.messages.create(**label_params(cases[c], gens[c]['text']))), len(sentences(gens[c]['text'])))
            if ok: labels[c] = labs
        except Exception as e: print(f'  {c}: {type(e).__name__}'); break
    print(f'{split} 정답: {len(labels)}/{len(valid)} 확보')
    sp = os.path.join(M1_DIR, f'scores_{split}.json')
    scores = load_json(sp) or {}
    refs = {}
    t0 = time.time()
    for k, c in enumerate([c for c in valid if c not in scores]):
        r = cases[c]['ref_judge']
        if r not in refs: refs = {r: (jF.prepare(r), jI.prepare(r))}          # 근거별로 연속 (같은 근거 두 모델)
        scores[c] = [list(p) for p in score_response(jF, jI, refs[r], gens[c]['text'], FN)]
        if (k + 1) % 25 == 0:
            save_json(scores, sp); print(f'  판정 {len(scores)}/{len(valid)} ({time.time() - t0:.0f}s)', flush=True)
    save_json(scores, sp)
    return gens, labels, scores

RUN = {'devL': run_split('devL')}

## 6. 사전 기준 판정 — M1-0, M1-1

In [ ]:
import pandas as pd

def wilson(k, n, z=1.96):
    if n == 0: return float('nan'), float('nan'), float('nan')
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

def conformal(maxes, alpha=0.10):
    # cal-L 노트북(conformal_flag)과 같은 등각 보정: 멀쩡한 응답 최대 점수의 ceil((n+1)(1-α))번째 + 1e-6
    mx = np.sort(np.asarray(maxes, float)); n = len(mx); k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(mx[k - 1]) + 1e-6 if k <= n else float('inf')

def response_rows(cases, gens, labels, scores, thr_c, thr_f):
    rows = []
    for cid, c in cases.items():
        g = gens.get(cid); st = gen_status(g)
        r = {'cid': cid, 'sid': c['sid'], 'task': c['task'], 'gen': c['gen'], 'flow': c['flow'], 'gold_in': c['gold_in'],
             'stop': g['stop'] if g else None, 'status': st}
        if st == 'valid':
            ss = sentences(g['text']); lab = labels.get(cid)
            r['status'] = 'ok' if lab is not None else 'nolabel'
            pts = scores.get(cid, [])
            r['n_sent'], r['n_pts'] = len(ss), len(pts)
            r['max_c'] = max([p[4] for p in pts], default=-np.inf)
            r['max_f'] = max([p[2] for p in pts], default=-np.inf)
            r['flag_c'], r['flag_f'] = bool(r['max_c'] >= thr_c), bool(r['max_f'] >= thr_f)
            first = next((p[0] for p in pts if p[4] >= thr_c), None)
            r['first_alarm_a'] = first
            if lab is not None:
                ls = [lab[i + 1]['label'] for i in range(len(ss))]
                r['n_H'], r['n_A'] = ls.count('H'), ls.count('A')
                r['clean'], r['strict_clean'] = r['n_H'] == 0, all(l == 'S' for l in ls)
                hs = [ss[i][0] for i, l in enumerate(ls) if l == 'H']
                r['first_H_a'] = hs[0] if hs else None
                r['strict_det'] = bool(hs and first is not None and first >= hs[0])
        rows.append(r)
    return pd.DataFrame(rows)

def rate_line(df, col='flag_c'):
    k, n = int(df[col].sum()), len(df)
    p, lo, hi = wilson(k, n)
    return {'k': k, 'n': n, 'rate': p, 'lo': lo, 'hi': hi}

def fmt(d):
    return f"{d['rate']:.3f} [{d['lo']:.3f}, {d['hi']:.3f}] ({d['k']}/{d['n']})" if d['n'] else '—'

def report(df, thr_name=''):
    ok = df[df.status == 'ok'].copy()
    for col in ['clean', 'strict_clean', 'flag_c', 'flag_f', 'strict_det']:
        ok[col] = ok[col].astype(bool)
    clean, hal = ok[ok.clean], ok[~ok.clean]
    out = {'status': df.status.value_counts().to_dict(), 'max_tokens_stops': int((df.stop == 'max_tokens').sum()),
           'clean_combo': rate_line(clean), 'clean_f1': rate_line(clean, 'flag_f'),
           'strict_clean_combo': rate_line(ok[ok.strict_clean]),
           'hal_any': rate_line(hal), 'hal_strict': rate_line(hal, 'strict_det'),
           'flag_precision_H': rate_line(ok[ok.flag_c], 'clean').copy(),
           'by_gen': {g: rate_line(d) for g, d in clean.groupby('gen')},
           'by_task': {t: rate_line(d) for t, d in clean.groupby('task')},
           'by_gen_task': {f'{g}|{t}': rate_line(d) for (g, t), d in clean.groupby(['gen', 'task'])},
           'hal_rate_by_gen': {g: float((~d.clean).mean()) for g, d in ok.groupby('gen')}}
    fp = out['flag_precision_H']                     # 표시된 응답 중 멀쩡한 비율 → H 비율로 바꿈
    if fp['n']: fp.update(k=fp['n'] - fp['k'], rate=1 - fp['rate'], lo=1 - fp['hi'], hi=1 - fp['lo'])
    qa = df[(df.flow == 'collection') & (df.gen == 'haiku')]
    out['qa_gold_in'] = float(qa.gold_in.astype(float).mean()) if len(qa) else None
    return out

DEV = response_rows(CASES['devL'], *RUN['devL'], THR_C, THR_F)
REP = {'devL': report(DEV)}
r = REP['devL']
valid_n = int(DEV.status.isin(['ok', 'nolabel']).sum()); ok_n = int((DEV.status == 'ok').sum())
label_ratio = ok_n / max(valid_n, 1)
M1_0 = bool(label_ratio >= 0.90 and r['clean_combo']['n'] >= 200)
M1_1 = bool(M1_0 and r['clean_combo']['rate'] <= 0.15)
print('응답 상태', r['status'], '| max_tokens로 끊긴 응답', r['max_tokens_stops'])
print(f"QA 검색에 원래 근거가 들어간 비율 {r['qa_gold_in']:.3f} | 환각 응답 비율(Opus H)", {g: round(v, 3) for g, v in r['hal_rate_by_gen'].items()})
print(f"\n멀쩡한 응답 표시율 (조합, 임계값 {THR_C:.3f}): {fmt(r['clean_combo'])}")
for g, d in r['by_gen'].items(): print(f'  {g:7} {fmt(d)}')
for t, d in r['by_task'].items(): print(f'  {t:9} {fmt(d)}')
for gt, d in r['by_gen_task'].items(): print(f'  {gt:18} {fmt(d)}')
print(f"보고: F1 base 단독(임계값 {THR_F:.3f}) {fmt(r['clean_f1'])} | 엄격 멀쩡(모두 S) {fmt(r['strict_clean_combo'])}")
print(f"보고: 환각 응답 탐지 아무 지점 {fmt(r['hal_any'])} / 첫 H 문장 이후 {fmt(r['hal_strict'])} | 표시된 응답 중 H 응답 {fmt(r['flag_precision_H'])}")
print(f"\n=== M1-0 유효성: 정답 확보 {label_ratio:.3f} (≥ 0.90), 멀쩡한 응답 {r['clean_combo']['n']} (≥ 200) → {'충족' if M1_0 else '미달 → 판정 보류'} ===")
print(f"=== M1-1: 멀쩡한 응답 표시율 {r['clean_combo']['rate']:.3f} (≤ 0.15) → {'충족' if M1_1 else ('미달' if M1_0 else '판정 보류')} ===")

## 7. M1-2 — M1-1 미달일 때만: cal-L로 임계값 재보정 → dev-L 확인

In [ ]:
M1_2, THR_NEW = None, None
if False:                    # m1_fresh: 재보정(M1-2) 없음
    CASES['calL'] = prepare_split('calL')
    RUN['calL'] = run_split('calL')
    CAL = response_rows(CASES['calL'], *RUN['calL'], THR_C, THR_F)
    cal_clean = CAL[(CAL.status == 'ok') & CAL.clean]
    THR_NEW = conformal(cal_clean.max_c)
    DEV2 = response_rows(CASES['devL'], *RUN['devL'], THR_NEW, THR_F)
    REP['calL'], REP['devL_recal'] = report(CAL), report(DEV2)
    r2 = REP['devL_recal']
    M1_2 = bool(r2['clean_combo']['rate'] <= 0.15)
    print(f"cal-L 멀쩡한 응답 {len(cal_clean)}개 | 기존 임계값에서 cal-L 표시율 {fmt(REP['calL']['clean_combo'])}")
    print(f'재보정 임계값 {THR_NEW:.4f} (기존 {THR_C:.4f})')
    print(f"dev-L 재보정: 멀쩡한 응답 표시율 {fmt(r2['clean_combo'])} | 환각 응답 탐지 {fmt(r2['hal_any'])} / 첫 H 이후 {fmt(r2['hal_strict'])}")
    print(f"=== M1-2: {'재보정 임계값 후보' if M1_2 else '임계값 재보정으로 해결 안 됨'} ===")
else:
    print('M1-2 실행 안 함 (M1-1 충족 또는 판정 보류)')

## 8. 저장

In [ ]:
usage = {}
for f in os.listdir(M1_DIR):
    if f.startswith(('gen_', 'label_')) and f.endswith('.json') and '_batches' not in f:
        usage[f] = round(cost(load_json(os.path.join(M1_DIR, f))), 2)
summ = {'gpu': GPU, 'thr_combo': THR_C, 'thr_f1': THR_F, 'combo': COMBO, 'gens': GENS, 'labeler': LABELER,
        'top_k': TOP_K, 'chunk_words': CHUNK_W, 'overlap': OVERLAP, 'n_sources': {k: len(v) for k, v in SIDS.items()},
        'label_ratio': label_ratio, 'M1_0': M1_0, 'M1_1': M1_1, 'M1_2': M1_2, 'thr_recal': THR_NEW,
        'report': REP, 'cost_usd': usage}
LOCAL = '/content/m1_fresh_local'; os.makedirs(LOCAL, exist_ok=True)
json.dump(summ, open(os.path.join(LOCAL, 'm1_summary.json'), 'w'), ensure_ascii=False, indent=1, default=float)
DEV.to_csv(os.path.join(LOCAL, 'm1_devL.csv'), index=False)
if 'calL' in RUN: CAL.to_csv(os.path.join(LOCAL, 'm1_calL.csv'), index=False)
for f in os.listdir(LOCAL):
    try: shutil.copy(os.path.join(LOCAL, f), os.path.join(M1_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)
print('API 사용량(추정 $)', usage)
zp = shutil.make_archive('/content/m1_fresh_results', 'zip', LOCAL)
from google.colab import files
files.download(zp)